[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-4_executed.ipynb)

---



# 93. The Inventory-Routing Problem (IRP)
## Tier 4: GAN-Based Demand Scenario Generation

### Purpose and workflow

An inventory-routing problem (IRP) combines replenishment decisions with vehicle routing: inventory must be delivered to customers over time, while delivery routes and inventory-related costs are considered together. This notebook demonstrates a data-driven uncertainty workflow. A generative adversarial network (GAN) learns from historical customer demand observations, generates demand scenarios, and supplies those scenarios to a scenario-based evaluation of a simplified IRP replenishment heuristic.

The workflow is:

1. Construct a small depot, customer, vehicle, and historical-demand instance.
2. Normalize the historical demand matrix and inspect its time series, distributions, and customer correlations.
3. Train a generator and discriminator adversarially using the normalized observations.
4. Generate and denormalize demand scenarios, then inspect their statistical properties.
5. Evaluate the same stated replenishment heuristic under each scenario and summarize the resulting cost distribution.

### GAN intuition and mathematical objective

The generator, denoted by $G$, maps a random noise vector $z$ to a synthetic demand vector. The discriminator, denoted by $D$, estimates whether a demand vector came from the historical data rather than the generator. The discriminator learns to assign high scores to historical observations and low scores to generated observations; the generator learns to make generated observations receive high discriminator scores.

For historical demand vectors $x$ drawn from the empirical data distribution $p_{\text{data}}$ and noise vectors $z$ drawn from a standard normal distribution $p_z$, the original minimax GAN objective is

$$
\min_G\max_D\;\mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))].
$$

The discriminator is trained with binary cross-entropy. The generator uses the non-saturating loss $-\mathbb{E}_{z\sim p_z}[\log D(G(z))]$, which gives a useful learning signal when the discriminator initially rejects generated samples. Both networks in this notebook are small, fully connected neural networks implemented with NumPy; their weights are updated using gradients computed by backpropagation.

### Scope and modeling assumptions

- The example has one depot, 10 customers, two nominal vehicles, 100 historical daily observations, and a five-period planning horizon. A generated scenario contains one demand value per customer; that same value is used in each of the five periods. This is a deliberately simple scenario structure, not a model of independent daily demand paths.
- Historical observations are generated synthetically for teaching purposes. They are not measured operational data, and the notebook does not establish that a trained GAN reproduces real-world demand.
- The IRP component is a transparent greedy heuristic, not an exact optimization model. It uses the first listed vehicle, prioritizes customers below a stated inventory threshold, and limits deliveries by vehicle capacity, depot stock, and customer storage capacity. The second vehicle is part of the example data but is not used by this heuristic.
- Scenario costs are evaluated independently. The reported mean and worst observed scenario costs describe this finite generated sample; they are not guarantees, confidence bounds, or the result of optimizing a single policy against all scenarios.
- The simplified inventory model does not include lead times, vehicle scheduling across periods, lost-sales or backorder accounting, or uncertainty in travel times. Negative inventory is prevented; unmet demand is represented by a stockout penalty when inventory falls below the customer's minimum level.

### What to examine

Inspect whether the generator's per-customer means and standard deviations resemble those of the normalized historical observations, and examine the generated scenarios and loss curves. These checks are descriptive rather than proof of scenario quality: matching marginal means and standard deviations alone does not establish that correlations, temporal structure, tails, or operationally important events have been learned. Then interpret the IRP cost summaries as outputs of the stated heuristic under the generated sample, not as evidence that GAN-based decisions outperform another method.

### Example instance and motivation

The example represents one depot, 10 customers, two vehicles with capacity 50 units each, a five-period horizon, and 100 historical demand observations per customer. The constructed observations include a weekly seasonal component, a trend component, customer-specific variation, and noise. A GAN is relevant when historical data contain dependencies or patterns that a simple independent-demand assumption may not represent. Its usefulness must still be assessed with appropriate validation and operational tests.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import itertools
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class DemandData:
    """Historical demand observations and their summary statistics."""

    customer_id: int
    demands: List[float]
    mean_demand: float
    std_demand: float


@dataclass
class GANConfig:
    """Dimensions and training settings for the demand GAN."""

    input_dim: int = 10
    hidden_dim: int = 64
    output_dim: int = 10
    lr_g: float = 0.0002
    lr_d: float = 0.0002
    epochs: int = 2000
    batch_size: int = 32


class DemandGenerator:
    """Fully connected generator mapping noise to normalized demands."""

    def __init__(self, config: GANConfig):
        self.config = config
        rng = np.random.RandomState(42)
        scale = 0.1

        self.W1 = rng.randn(config.input_dim, config.hidden_dim) * scale
        self.b1 = np.zeros(config.hidden_dim)
        self.W2 = rng.randn(config.hidden_dim, config.hidden_dim) * scale
        self.b2 = np.zeros(config.hidden_dim)
        self.W3 = rng.randn(config.hidden_dim, config.output_dim) * scale
        self.b3 = np.zeros(config.output_dim)

    @staticmethod
    def relu(values: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, values)

    def forward(self, noise: np.ndarray) -> np.ndarray:
        """Map each noise vector to one normalized demand vector."""
        hidden_1 = self.relu(noise @ self.W1 + self.b1)
        hidden_2 = self.relu(hidden_1 @ self.W2 + self.b2)
        return hidden_2 @ self.W3 + self.b3

    def forward_with_cache(self, noise: np.ndarray):
        """Return the forward-pass values needed for backpropagation."""
        pre_activation_1 = noise @ self.W1 + self.b1
        hidden_1 = self.relu(pre_activation_1)
        pre_activation_2 = hidden_1 @ self.W2 + self.b2
        hidden_2 = self.relu(pre_activation_2)
        output = hidden_2 @ self.W3 + self.b3

        cache = (noise, pre_activation_1, hidden_1, pre_activation_2, hidden_2)
        return output, cache

    def backward(self, output_gradient: np.ndarray, cache):
        """Calculate gradients of the generator loss with respect to its weights."""
        noise, pre_activation_1, hidden_1, pre_activation_2, hidden_2 = cache
        batch_size = noise.shape[0]

        grad_W3 = hidden_2.T @ output_gradient
        grad_b3 = np.sum(output_gradient, axis=0)

        grad_hidden_2 = output_gradient @ self.W3.T
        grad_pre_activation_2 = grad_hidden_2 * (pre_activation_2 > 0)
        grad_W2 = hidden_1.T @ grad_pre_activation_2
        grad_b2 = np.sum(grad_pre_activation_2, axis=0)

        grad_hidden_1 = grad_pre_activation_2 @ self.W2.T
        grad_pre_activation_1 = grad_hidden_1 * (pre_activation_1 > 0)
        grad_W1 = noise.T @ grad_pre_activation_1
        grad_b1 = np.sum(grad_pre_activation_1, axis=0)

        return {
            "W1": grad_W1 / batch_size,
            "b1": grad_b1 / batch_size,
            "W2": grad_W2 / batch_size,
            "b2": grad_b2 / batch_size,
            "W3": grad_W3 / batch_size,
            "b3": grad_b3 / batch_size,
        }

    def apply_gradients(self, gradients: Dict[str, np.ndarray], learning_rate: float):
        """Apply one gradient-descent update to every generator parameter."""
        for parameter_name, gradient in gradients.items():
            parameter = getattr(self, parameter_name)
            parameter -= learning_rate * gradient


class DemandDiscriminator:
    """Fully connected discriminator returning real-demand probabilities."""

    def __init__(self, config: GANConfig):
        self.config = config
        rng = np.random.RandomState(123)
        scale = 0.1

        self.W1 = rng.randn(config.output_dim, config.hidden_dim) * scale
        self.b1 = np.zeros(config.hidden_dim)
        self.W2 = rng.randn(config.hidden_dim, config.hidden_dim) * scale
        self.b2 = np.zeros(config.hidden_dim)
        self.W3 = rng.randn(config.hidden_dim, 1) * scale
        self.b3 = np.zeros(1)

    @staticmethod
    def relu(values: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, values)

    @staticmethod
    def sigmoid(values: np.ndarray) -> np.ndarray:
        clipped = np.clip(values, -500.0, 500.0)
        return 1.0 / (1.0 + np.exp(-clipped))

    def forward(self, demands: np.ndarray) -> np.ndarray:
        """Return the discriminator's estimated probability of real data."""
        hidden_1 = self.relu(demands @ self.W1 + self.b1)
        hidden_2 = self.relu(hidden_1 @ self.W2 + self.b2)
        logits = hidden_2 @ self.W3 + self.b3
        return self.sigmoid(logits)

    def forward_with_cache(self, demands: np.ndarray):
        """Return probabilities and intermediate values needed for gradients."""
        pre_activation_1 = demands @ self.W1 + self.b1
        hidden_1 = self.relu(pre_activation_1)
        pre_activation_2 = hidden_1 @ self.W2 + self.b2
        hidden_2 = self.relu(pre_activation_2)
        logits = hidden_2 @ self.W3 + self.b3
        probabilities = self.sigmoid(logits)

        cache = (demands, pre_activation_1, hidden_1, pre_activation_2, hidden_2)
        return probabilities, cache

    def backward(self, output_gradient: np.ndarray, cache):
        """Backpropagate a gradient with respect to discriminator logits."""
        demands, pre_activation_1, hidden_1, pre_activation_2, hidden_2 = cache
        batch_size = demands.shape[0]

        grad_W3 = hidden_2.T @ output_gradient
        grad_b3 = np.sum(output_gradient, axis=0)

        grad_hidden_2 = output_gradient @ self.W3.T
        grad_pre_activation_2 = grad_hidden_2 * (pre_activation_2 > 0)
        grad_W2 = hidden_1.T @ grad_pre_activation_2
        grad_b2 = np.sum(grad_pre_activation_2, axis=0)

        grad_hidden_1 = grad_pre_activation_2 @ self.W2.T
        grad_pre_activation_1 = grad_hidden_1 * (pre_activation_1 > 0)
        grad_W1 = demands.T @ grad_pre_activation_1
        grad_b1 = np.sum(grad_pre_activation_1, axis=0)
        grad_demands = grad_pre_activation_1 @ self.W1.T

        gradients = {
            "W1": grad_W1 / batch_size,
            "b1": grad_b1 / batch_size,
            "W2": grad_W2 / batch_size,
            "b2": grad_b2 / batch_size,
            "W3": grad_W3 / batch_size,
            "b3": grad_b3 / batch_size,
        }
        return grad_demands, gradients

    def apply_gradients(self, gradients: Dict[str, np.ndarray], learning_rate: float):
        """Apply one gradient-descent update to every discriminator parameter."""
        for parameter_name, gradient in gradients.items():
            parameter = getattr(self, parameter_name)
            parameter -= learning_rate * gradient

In [ ]:
class DemandGAN:
    """Train a generator and discriminator on customer demand vectors."""

    def __init__(self, config: GANConfig):
        self.config = config
        self.generator = DemandGenerator(config)
        self.discriminator = DemandDiscriminator(config)
        self.rng = np.random.RandomState(42)

        self.generator_loss_history = []
        self.discriminator_loss_history = []

    def generate_noise(self, batch_size: int) -> np.ndarray:
        """Draw reproducible standard-normal noise vectors for the generator."""
        return self.rng.randn(batch_size, self.config.input_dim)

    @staticmethod
    def generator_loss(fake_output: np.ndarray) -> float:
        """Non-saturating generator binary cross-entropy loss."""
        return float(-np.mean(np.log(fake_output + 1e-8)))

    @staticmethod
    def discriminator_loss(real_output: np.ndarray, fake_output: np.ndarray) -> float:
        """Binary cross-entropy for real and generated observations."""
        real_loss = -np.mean(np.log(real_output + 1e-8))
        fake_loss = -np.mean(np.log(1.0 - fake_output + 1e-8))
        return float(real_loss + fake_loss)

    def update_generator(self, noise: np.ndarray):
        """Update the generator using gradients through the fixed discriminator."""
        fake_demands, generator_cache = self.generator.forward_with_cache(noise)
        fake_output, discriminator_cache = self.discriminator.forward_with_cache(
            fake_demands
        )

        # For the non-saturating loss, the logit gradient is (D(fake) - 1) / batch.
        logit_gradient = (fake_output - 1.0) / fake_output.shape[0]
        demand_gradient, _ = self.discriminator.backward(
            logit_gradient, discriminator_cache
        )
        generator_gradients = self.generator.backward(demand_gradient, generator_cache)
        self.generator.apply_gradients(generator_gradients, self.config.lr_g)
        return self.generator_loss(fake_output)

    def update_discriminator(self, real_demands: np.ndarray, noise: np.ndarray):
        """Update the discriminator to distinguish historical and generated data."""
        fake_demands = self.generator.forward(noise)
        real_output, real_cache = self.discriminator.forward_with_cache(real_demands)
        fake_output, fake_cache = self.discriminator.forward_with_cache(fake_demands)

        real_gradient, real_parameter_gradients = self.discriminator.backward(
            real_output - 1.0, real_cache
        )
        fake_gradient, fake_parameter_gradients = self.discriminator.backward(
            fake_output, fake_cache
        )
        del real_gradient, fake_gradient

        combined_gradients = {
            name: real_parameter_gradients[name] + fake_parameter_gradients[name]
            for name in real_parameter_gradients
        }
        self.discriminator.apply_gradients(combined_gradients, self.config.lr_d)

        loss = self.discriminator_loss(real_output, fake_output)
        return loss

    def train(self, real_demands: np.ndarray):
        """Train both networks, including any final partial batch."""
        real_demands = np.asarray(real_demands, dtype=float)
        if real_demands.ndim != 2 or real_demands.shape[1] != self.config.output_dim:
            raise ValueError("The demand matrix must have one column per GAN output.")
        if real_demands.shape[0] == 0:
            raise ValueError("At least one historical demand observation is required.")
        if self.config.batch_size <= 0 or self.config.epochs <= 0:
            raise ValueError("Batch size and epoch count must be positive.")

        print(f"Training GAN for {self.config.epochs} epochs...")
        num_samples = real_demands.shape[0]

        for epoch in range(self.config.epochs):
            permutation = self.rng.permutation(num_samples)
            epoch_g_loss = 0.0
            epoch_d_loss = 0.0
            num_batches = 0

            for start_idx in range(0, num_samples, self.config.batch_size):
                batch_indices = permutation[start_idx:start_idx + self.config.batch_size]
                real_batch = real_demands[batch_indices]
                batch_size = real_batch.shape[0]

                noise_for_discriminator = self.generate_noise(batch_size)
                d_loss = self.update_discriminator(real_batch, noise_for_discriminator)

                noise_for_generator = self.generate_noise(batch_size)
                g_loss = self.update_generator(noise_for_generator)

                epoch_g_loss += g_loss
                epoch_d_loss += d_loss
                num_batches += 1

            avg_g_loss = epoch_g_loss / num_batches
            avg_d_loss = epoch_d_loss / num_batches
            self.generator_loss_history.append(avg_g_loss)
            self.discriminator_loss_history.append(avg_d_loss)

            if (epoch + 1) % 200 == 0:
                print(
                    f"Epoch {epoch + 1}: G_loss = {avg_g_loss:.4f}, "
                    f"D_loss = {avg_d_loss:.4f}"
                )

        print("GAN training completed.")

    def generate_scenarios(self, num_scenarios: int) -> np.ndarray:
        """Generate normalized demand vectors from fresh noise."""
        if num_scenarios <= 0:
            raise ValueError("The number of scenarios must be positive.")
        noise = self.generate_noise(num_scenarios)
        return self.generator.forward(noise)

In [ ]:
class RobustIRPOptimizer:
    """Evaluate a greedy replenishment heuristic across demand scenarios."""

    def __init__(self, depot, customers, vehicles, num_periods: int):
        if not customers:
            raise ValueError("At least one customer is required.")
        if not vehicles:
            raise ValueError("At least one vehicle is required.")
        if num_periods <= 0:
            raise ValueError("The planning horizon must be positive.")

        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.distances = self._compute_distances()

    def _compute_distances(self) -> Dict[Tuple[int, int], float]:
        """Precompute Euclidean distances between the depot and customers."""
        distances = {}

        for customer in self.customers:
            distance = np.hypot(self.depot.x - customer.x, self.depot.y - customer.y)
            distances[(0, customer.id)] = distance
            distances[(customer.id, 0)] = distance

        for customer_1 in self.customers:
            for customer_2 in self.customers:
                if customer_1.id != customer_2.id:
                    distance = np.hypot(
                        customer_1.x - customer_2.x,
                        customer_1.y - customer_2.y,
                    )
                    distances[(customer_1.id, customer_2.id)] = distance

        return distances

    def _route_distance(self, customers: List[int]) -> float:
        """Measure a closed route that starts and ends at the depot."""
        if not customers:
            return 0.0

        total_distance = self.distances[(0, customers[0])]
        for index in range(len(customers) - 1):
            total_distance += self.distances[(customers[index], customers[index + 1])]
        total_distance += self.distances[(customers[-1], 0)]
        return total_distance

    def _solve_irp_scenario(self, demand_scenario: np.ndarray) -> float:
        """Evaluate one customer-demand vector over the planning horizon."""
        demand_scenario = np.asarray(demand_scenario, dtype=float)
        if demand_scenario.shape != (len(self.customers),):
            raise ValueError("Each scenario must contain one demand value per customer.")
        if np.any(demand_scenario < 0):
            raise ValueError("Demand values must be non-negative.")

        total_cost = 0.0
        inventories = {0: float(self.depot.initial_inventory)}
        for customer in self.customers:
            inventories[customer.id] = float(customer.initial_inventory)

        # This teaching heuristic uses the first vehicle and one route per period.
        vehicle = self.vehicles[0]

        for _period in range(self.num_periods):
            urgent_customers = [
                customer
                for customer in self.customers
                if inventories[customer.id] < 2.0 * customer.min_inventory
            ]
            urgent_customers.sort(key=lambda customer: inventories[customer.id])

            remaining_vehicle_capacity = float(vehicle.capacity)
            route_customers = []
            deliveries = []

            for customer in urgent_customers:
                available_customer_space = max(
                    0.0, customer.max_inventory - inventories[customer.id]
                )
                delivery = min(
                    20.0,
                    remaining_vehicle_capacity,
                    inventories[0],
                    available_customer_space,
                )

                if delivery > 0.0:
                    route_customers.append(customer.id)
                    deliveries.append((customer.id, delivery))
                    remaining_vehicle_capacity -= delivery
                    inventories[0] -= delivery
                    inventories[customer.id] += delivery

                if remaining_vehicle_capacity <= 0.0 or inventories[0] <= 0.0:
                    break

            if route_customers:
                route_cost = vehicle.fixed_cost + vehicle.variable_cost * self._route_distance(
                    route_customers
                )
                total_cost += route_cost

            for customer, demand in zip(self.customers, demand_scenario):
                inventories[customer.id] = max(
                    0.0, inventories[customer.id] - demand
                )

            total_cost += max(0.0, inventories[0]) * self.depot.holding_cost
            for customer in self.customers:
                total_cost += inventories[customer.id] * customer.holding_cost
                if inventories[customer.id] < customer.min_inventory:
                    total_cost += 1000.0

        return float(total_cost)

    def solve_robust_irp(self, scenarios: np.ndarray) -> Tuple[float, Dict]:
        """Summarize heuristic costs over a collection of demand scenarios."""
        scenarios = np.asarray(scenarios, dtype=float)
        expected_shape = (scenarios.shape[0], len(self.customers)) if scenarios.ndim == 2 else None
        if expected_shape is None or scenarios.shape[0] == 0:
            raise ValueError("Scenarios must be a non-empty two-dimensional array.")
        if scenarios.shape != expected_shape:
            raise ValueError("Each scenario must contain one value per customer.")

        print(f"Evaluating the replenishment heuristic on {len(scenarios)} scenarios...")
        scenario_costs = []

        for index, scenario in enumerate(scenarios):
            scenario_costs.append(self._solve_irp_scenario(scenario))
            if (index + 1) % 100 == 0:
                print(f"  Evaluated {index + 1}/{len(scenarios)} scenarios")

        average_cost = float(np.mean(scenario_costs))
        worst_cost = float(np.max(scenario_costs))
        best_cost = float(np.min(scenario_costs))
        cost_std = float(np.std(scenario_costs))

        results = {
            "average_cost": average_cost,
            "worst_case_cost": worst_cost,
            "best_case_cost": best_cost,
            "cost_std": cost_std,
            "scenario_costs": scenario_costs,
        }

        print("\nScenario cost summary:")
        print(f"  Mean cost: ${average_cost:.2f}")
        print(f"  Worst observed cost: ${worst_cost:.2f}")
        print(f"  Best observed cost: ${best_cost:.2f}")
        print(f"  Cost standard deviation: ${cost_std:.2f}")

        return average_cost, results

## 1. Create the example instance

Each customer is assigned a location and a 100-observation demand history. The shared construction of the seasonal and trend components creates structured variation across days; a customer-specific component and random noise add further variation. The historical observations are stored together by day so that each row of the training matrix represents the demand vector observed across all customers on that day.

In [ ]:
def create_gan_instance():
    """Create the depot, customers, vehicles, and synthetic demand histories."""
    depot = type(
        "Depot",
        (),
        {
            "x": 0.0,
            "y": 0.0,
            "initial_inventory": 600.0,
            "holding_cost": 0.1,
        },
    )()

    rng = np.random.RandomState(42)
    customers = []
    base_demands = [12, 8, 15, 10, 18, 14, 9, 11, 13, 16]

    for index in range(10):
        angle = 2.0 * np.pi * index / 10
        distance = 15.0 + rng.normal(0.0, 3.0)
        x = distance * np.cos(angle)
        y = distance * np.sin(angle)
        historical_demands = []

        for day in range(100):
            seasonal = 1.0 + 0.3 * np.sin(2.0 * np.pi * day / 7.0)
            trend = 1.0 + 0.01 * day / 100.0
            customer_factor = 1.0 + 0.2 * np.sin(
                2.0 * np.pi * index / 10.0 + day / 20.0
            )
            demand = base_demands[index] * seasonal * trend * customer_factor
            demand += rng.normal(0.0, base_demands[index] * 0.1)
            historical_demands.append(max(3.0, demand))

        demand_mean = float(np.mean(historical_demands))
        demand_std = float(np.std(historical_demands))
        customer = type(
            "Customer",
            (),
            {
                "id": index + 1,
                "x": x,
                "y": y,
                "demand_mean": demand_mean,
                "demand_std": demand_std,
                "holding_cost": 0.1 + 0.02 * index,
                "initial_inventory": demand_mean * 3.0,
                "min_inventory": demand_mean * 1.5,
                "max_inventory": demand_mean * 6.0,
                "historical_demands": historical_demands,
            },
        )()
        customers.append(customer)

    vehicles = [
        type(
            "Vehicle",
            (),
            {"id": 1, "capacity": 50.0, "fixed_cost": 50.0, "variable_cost": 1.0},
        )(),
        type(
            "Vehicle",
            (),
            {"id": 2, "capacity": 50.0, "fixed_cost": 50.0, "variable_cost": 1.0},
        )(),
    ]

    return depot, customers, vehicles


print("Creating the example IRP instance with historical demand data...")
depot, customers, vehicles = create_gan_instance()

print(f"Depot: ({depot.x:.1f}, {depot.y:.1f}), inventory: {depot.initial_inventory:.1f}")
print("\nCustomers (first five):")
for customer in customers[:5]:
    print(
        f"  C{customer.id}: ({customer.x:.1f}, {customer.y:.1f}), "
        f"mean demand: {customer.demand_mean:.1f}, "
        f"standard deviation: {customer.demand_std:.1f}, "
        f"historical observations: {len(customer.historical_demands)}"
    )

print("\nVehicles:")
for vehicle in vehicles:
    print(
        f"  V{vehicle.id}: capacity {vehicle.capacity:.1f}, "
        f"fixed route cost ${vehicle.fixed_cost:.2f}"
    )

## 2. Prepare and inspect historical demand

Let $X_{t,i}$ denote demand for customer $i$ on historical day $t$. The training matrix has one row per day and one column per customer. Each customer column is standardized as $Z_{t,i}=(X_{t,i}-\mu_i)/\sigma_i$, where $\mu_i$ and $\sigma_i$ are that customer's historical mean and standard deviation. Standardization puts customers with different demand scales on comparable numerical scales during training; the stored means and standard deviations are used later to return generated values to the original demand units. A zero standard deviation is replaced with one to avoid division by zero.

In [ ]:
def prepare_demand_data(customers):
    """Build and standardize a day-by-customer historical demand matrix."""
    if not customers:
        raise ValueError("At least one customer is required.")

    observation_counts = {len(customer.historical_demands) for customer in customers}
    if len(observation_counts) != 1:
        raise ValueError("Every customer must have the same number of observations.")

    demand_array = np.asarray(
        [customer.historical_demands for customer in customers], dtype=float
    ).T
    demand_means = np.mean(demand_array, axis=0)
    demand_stds = np.std(demand_array, axis=0)
    safe_stds = np.where(demand_stds == 0.0, 1.0, demand_stds)
    normalized_demands = (demand_array - demand_means) / safe_stds

    return normalized_demands, demand_means, safe_stds


print("Preparing historical demand data for GAN training...")
normalized_demands, demand_means, demand_stds = prepare_demand_data(customers)
print(f"Demand matrix shape (days, customers): {normalized_demands.shape}")
print(f"Customer means: {demand_means}")
print(f"Customer standard deviations: {demand_stds}")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Historical Demand Data Analysis", fontsize=16, fontweight="bold")

sample_customer_indices = [0, 2, 4, 6]
for customer_index in sample_customer_indices:
    customer = customers[customer_index]
    axes[0, 0].plot(
        customer.historical_demands,
        label=f"C{customer.id}",
        alpha=0.8,
    )
axes[0, 0].set_title("Daily Demand for Sample Customers")
axes[0, 0].set_xlabel("Day")
axes[0, 0].set_ylabel("Demand (units)")
axes[0, 0].legend(title="Customer")
axes[0, 0].grid(True, alpha=0.3)

all_demands = np.concatenate(
    [np.asarray(customer.historical_demands) for customer in customers]
)
axes[0, 1].hist(
    all_demands,
    bins=30,
    alpha=0.75,
    color="skyblue",
    edgecolor="black",
)
axes[0, 1].set_title("Pooled Historical Demand Distribution")
axes[0, 1].set_xlabel("Demand (units)")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].grid(True, alpha=0.3)

demand_correlation = np.corrcoef(normalized_demands, rowvar=False)
image = axes[1, 0].imshow(
    demand_correlation,
    cmap="coolwarm",
    aspect="auto",
    vmin=-1.0,
    vmax=1.0,
)
axes[1, 0].set_title("Customer Demand Correlation")
axes[1, 0].set_xlabel("Customer")
axes[1, 0].set_ylabel("Customer")
axes[1, 0].set_xticks(range(len(customers)))
axes[1, 0].set_yticks(range(len(customers)))
axes[1, 0].set_xticklabels([f"C{i + 1}" for i in range(len(customers))])
axes[1, 0].set_yticklabels([f"C{i + 1}" for i in range(len(customers))])
fig.colorbar(image, ax=axes[1, 0], label="Pearson correlation")

customer_means = [customer.demand_mean for customer in customers]
customer_stds = [customer.demand_std for customer in customers]
customer_labels = [f"C{customer.id}" for customer in customers]
positions = np.arange(len(customer_labels))
axes[1, 1].bar(positions - 0.2, customer_means, 0.4, label="Mean", alpha=0.8)
axes[1, 1].bar(positions + 0.2, customer_stds, 0.4, label="Standard deviation", alpha=0.8)
axes[1, 1].set_title("Demand Statistics by Customer")
axes[1, 1].set_xlabel("Customer")
axes[1, 1].set_ylabel("Demand (units)")
axes[1, 1].set_xticks(positions)
axes[1, 1].set_xticklabels(customer_labels, rotation=45)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Train the GAN

For each mini-batch, the discriminator first receives historical and generated demand vectors. With discriminator output $D(u)$ for input vector $u$, its batch loss is the sum of the real-data loss $-\log D(x)$ and generated-data loss $-\log(1-D(G(z)))$. The generator is then updated to reduce $-\log D(G(z))$. Backpropagation applies the chain rule through the network layers, and gradient descent updates the weights and biases. The training curves are diagnostics, not a guarantee of convergence or realistic scenario generation.

In [ ]:
def train_demand_gan(real_demands):
    """Configure and train the GAN on standardized historical demand vectors."""
    gan_config = GANConfig(
        input_dim=10,
        hidden_dim=64,
        output_dim=10,
        lr_g=0.0002,
        lr_d=0.0002,
        epochs=1000,
        batch_size=16,
    )

    if real_demands.shape[1] != gan_config.output_dim:
        raise ValueError("GAN output_dim must equal the number of customers.")

    model = DemandGAN(gan_config)
    print("\nStarting GAN training...")
    start_time = time.time()
    model.train(real_demands)
    elapsed_time = time.time() - start_time
    print(f"GAN training elapsed time: {elapsed_time:.2f} seconds")
    return model


gan = train_demand_gan(normalized_demands)

## 4. Generate and inspect demand scenarios

The generator produces normalized vectors. For customer $i$, the inverse standardization is $\widehat{X}_i=\widehat{Z}_i\sigma_i+\mu_i$. Because demand is non-negative, values below one unit are floored at one unit for this example. This operational post-processing changes the generated distribution and should be included when interpreting the diagnostic statistics. Comparing means and standard deviations is a useful first check, but it does not test every feature of a multivariate distribution.

In [ ]:
def analyze_generated_scenarios(gan, demand_means, demand_stds, real_demands, customers):
    """Generate scenarios, report marginal diagnostics, and visualize samples."""
    num_scenarios = 500
    print("\nGenerating demand scenarios")
    print(f"Requested scenarios: {num_scenarios}")

    start_time = time.time()
    normalized_scenarios = gan.generate_scenarios(num_scenarios)
    scenarios = normalized_scenarios * demand_stds + demand_means
    scenarios = np.maximum(scenarios, 1.0)
    elapsed_time = time.time() - start_time
    print(f"Scenario generation elapsed time: {elapsed_time:.2f} seconds")

    real_means = np.mean(real_demands, axis=0)
    real_stds = np.std(real_demands, axis=0)
    generated_means = np.mean(normalized_scenarios, axis=0)
    generated_stds = np.std(normalized_scenarios, axis=0)
    mean_error = float(np.mean(np.abs(real_means - generated_means)))
    std_error = float(np.mean(np.abs(real_stds - generated_stds)))

    print("\nMarginal diagnostics in standardized units:")
    print(f"Mean absolute error across customer means: {mean_error:.4f}")
    print(f"Mean absolute error across customer standard deviations: {std_error:.4f}")

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("GAN-Generated Demand Scenario Diagnostics", fontsize=16, fontweight="bold")
    customer_labels = [f"C{customer.id}" for customer in customers]
    positions = np.arange(len(customer_labels))

    axes[0, 0].bar(positions - 0.2, real_means, 0.4, label="Historical", alpha=0.8)
    axes[0, 0].bar(positions + 0.2, generated_means, 0.4, label="Generated", alpha=0.8)
    axes[0, 0].set_title("Standardized Mean by Customer")
    axes[0, 0].set_xlabel("Customer")
    axes[0, 0].set_ylabel("Standardized demand")
    axes[0, 0].set_xticks(positions)
    axes[0, 0].set_xticklabels(customer_labels, rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].bar(positions - 0.2, real_stds, 0.4, label="Historical", alpha=0.8)
    axes[0, 1].bar(positions + 0.2, generated_stds, 0.4, label="Generated", alpha=0.8)
    axes[0, 1].set_title("Standardized Standard Deviation by Customer")
    axes[0, 1].set_xlabel("Customer")
    axes[0, 1].set_ylabel("Standardized demand")
    axes[0, 1].set_xticks(positions)
    axes[0, 1].set_xticklabels(customer_labels, rotation=45)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    epochs = np.arange(1, len(gan.generator_loss_history) + 1)
    axes[1, 0].plot(epochs, gan.generator_loss_history, label="Generator loss", alpha=0.8)
    axes[1, 0].plot(epochs, gan.discriminator_loss_history, label="Discriminator loss", alpha=0.8)
    axes[1, 0].set_title("GAN Training Loss")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Binary cross-entropy loss")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    for scenario_index, scenario in enumerate(scenarios[:5]):
        axes[1, 1].plot(
            customer_labels,
            scenario,
            marker="o",
            alpha=0.75,
            label=f"Scenario {scenario_index + 1}",
        )
    axes[1, 1].set_title("Sample Generated Scenarios")
    axes[1, 1].set_xlabel("Customer")
    axes[1, 1].set_ylabel("Demand (units per period)")
    axes[1, 1].tick_params(axis="x", rotation=45)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return scenarios


scenarios = analyze_generated_scenarios(
    gan,
    demand_means,
    demand_stds,
    normalized_demands,
    customers,
)

## 5. Evaluate the IRP heuristic across scenarios

For each demand vector, the heuristic starts from the same initial inventory state and simulates the five periods. At each period it identifies customers below twice their minimum inventory, orders them by current inventory, and considers deliveries of at most 20 units per customer. A delivery is also limited by remaining vehicle capacity, depot inventory, and customer storage space. The route cost is the vehicle's fixed cost plus its variable cost multiplied by the closed Euclidean route distance. Demand is then applied, holding costs are charged on remaining inventory, and a fixed penalty is charged for each customer below its minimum inventory.

If $C_s$ is the resulting cost for generated scenario $s$, the notebook reports the sample mean $\frac{1}{S}\sum_{s=1}^{S}C_s$, the largest observed value $\max_s C_s$, the smallest observed value $\min_s C_s$, and the population standard deviation. These are descriptive scenario-evaluation statistics for this heuristic; no optimization over a common cross-scenario policy is performed.

In [ ]:
def solve_robust_irp_with_scenarios(depot, customers, vehicles, scenarios):
    """Evaluate the greedy IRP heuristic on GAN-generated demand scenarios."""
    print("\nScenario-based IRP heuristic evaluation")
    optimizer = RobustIRPOptimizer(
        depot=depot,
        customers=customers,
        vehicles=vehicles,
        num_periods=5,
    )

    start_time = time.time()
    average_cost, results = optimizer.solve_robust_irp(scenarios)
    elapsed_time = time.time() - start_time
    print(f"Scenario evaluation elapsed time: {elapsed_time:.2f} seconds")
    return average_cost, results


robust_cost, robust_results = solve_robust_irp_with_scenarios(
    depot,
    customers,
    vehicles,
    scenarios,
)

## 6. Compare with a mean-demand reference

A deterministic reference replaces each customer's uncertain demand with its historical mean and evaluates the same heuristic on that single demand vector. This is a point-estimate benchmark, not a separate routing policy. Its cost is not directly comparable to the mean scenario cost as if both were costs of a shared decision under uncertainty: the reference contains one mean-demand case, whereas the scenario statistic averages many generated cases. The plots make that distinction visible and allow inspection of the scenario cost spread.

In [ ]:
def compare_with_deterministic(
    depot,
    customers,
    vehicles,
    scenarios,
    scenario_results,
):
    """Compare scenario costs with the heuristic's historical-mean reference case."""
    print("\nMean-demand reference and scenario evaluation")
    mean_demands = np.asarray(
        [customer.demand_mean for customer in customers], dtype=float
    )
    optimizer = RobustIRPOptimizer(depot, customers, vehicles, num_periods=5)
    mean_demand_cost = optimizer._solve_irp_scenario(mean_demands)

    scenario_costs = np.asarray(scenario_results["scenario_costs"], dtype=float)
    average_scenario_cost = float(np.mean(scenario_costs))
    worst_scenario_cost = float(np.max(scenario_costs))
    scenario_cost_std = float(np.std(scenario_costs))

    print(f"Historical-mean reference cost: ${mean_demand_cost:.2f}")
    print(f"Mean generated-scenario cost: ${average_scenario_cost:.2f}")
    print(f"Worst observed generated-scenario cost: ${worst_scenario_cost:.2f}")
    print(f"Generated-scenario cost standard deviation: ${scenario_cost_std:.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].bar(
        ["Historical-mean\nreference", "Mean of generated\nscenarios"],
        [mean_demand_cost, average_scenario_cost],
        color=["steelblue", "indianred"],
        alpha=0.8,
    )
    axes[0].set_title("Heuristic Cost Summaries")
    axes[0].set_ylabel("Cost (dollar units)")
    axes[0].grid(True, axis="y", alpha=0.3)

    axes[1].hist(
        scenario_costs,
        bins=25,
        alpha=0.75,
        color="indianred",
        edgecolor="black",
        label="Generated-scenario costs",
    )
    axes[1].axvline(
        mean_demand_cost,
        color="steelblue",
        linestyle="--",
        linewidth=2,
        label="Historical-mean reference",
    )
    axes[1].set_title("Generated-Scenario Cost Distribution")
    axes[1].set_xlabel("Cost (dollar units)")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return mean_demand_cost


mean_demand_reference_cost = compare_with_deterministic(
    depot,
    customers,
    vehicles,
    scenarios,
    robust_results,
)

## Interpretation and limitations

The notebook links demand modeling to inventory-routing evaluation in a sequence of explicit steps: historical observations are standardized, a GAN is trained by adversarial gradient updates, generated vectors are returned to demand units, and each vector is evaluated by a stated replenishment heuristic. The output can be used to inspect how that heuristic's costs vary over the generated sample.

Several distinctions are important when interpreting the results:

- GAN loss values measure the networks' binary-classification objectives. Their values alone do not establish convergence, diversity, statistical fidelity, or operational usefulness.
- The marginal mean and standard-deviation diagnostics do not measure all dependence structure. A useful assessment would also examine joint and tail behavior and compare generated scenarios with held-out observations.
- The historical data in this example are synthetically constructed. Performance on them cannot establish performance on a real supply chain.
- The IRP component is a greedy heuristic. It does not solve an exact inventory-routing optimization model, and the second vehicle is not dispatched.
- The average and worst observed costs depend on the finite generated scenario sample. They are not formal robustness guarantees, and the notebook does not optimize one decision plan across scenarios.
- The historical-mean reference is one deterministic input case. Comparing it with a distribution of scenario costs is descriptive and does not by itself establish that either approach is superior.

In practice, the modeling assumptions, demand-generation quality, inventory and routing constraints, and operational costs should be validated against the intended application. Natural extensions within this workflow include representing multiple periods of demand in each scenario, modeling additional operational constraints, and evaluating the method on held-out historical observations.